## Exercise 4: Data Warehouse Querying and Basic Geospatial Operations

Skills: 
* Query data warehouse table
* Use dictionary to map values

References: 
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-basics.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intro.html
* https://cityoflosangeles.github.io/best-practices/spatial-analysis-intermediate.html
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [1]:
import geopandas as gpd
import pandas as pd
import os
import shared_utils


#os.environ["CALITP_BQ_MAX_BYTES"] = str(100_000_000_000)
pd.set_option("display.max_rows", 20)

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.9.1-CAPI-1.14.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


## Query a table, turn it into a gdf
<b> QUESTION </b> I guess you already did the stuff below for me ?

For two operators, `ITP_ID == 246` (Caltrain), `ITP_ID == 279` (BART) and `ITP_ID == 343` (Merced the Bus), you will query the warehouse table.

* Query `views.gtfs_schedule_dim_stops`
* Filter to the ITP_ID of interest
* Select these columns: `calitp_itp_id`, `stop_id`, `stop_lat`, `stop_lon`, `stop_name`
* Return as a dataframe using `collect()`
* Turn the point data into geometry with `geopandas`: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html)

<b> QUESTION </b> I think I do last step a few cells below? Or is it important I do it right at the beginning?

In [2]:
ITP_ID = [246, 279, 343]
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
)

## Use a dictionary to map values

* Create a new column called `operator` where `itp_id` is associated with its operator name.
* First, write a function to do it.
* Then, use a dictionary to do it (create new column called `agency`).
* Double check that `operator` and `agency` show the same values. Use `assert` to check.
    * `assert df.operator == df.agency`
* Hint: https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html

In [3]:
#previewing
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name
265688,246,2537740,37.438516,-122.156443,Stanford Caltrain
265689,246,2537740,37.438516,-122.156443,Stanford Caltrain Station


In [4]:
stops.dtypes

itp_id         int64
stop_id       object
stop_lat     float64
stop_lon     float64
stop_name     object
dtype: object

#### Function

In [5]:
def operator_names(x):
    if x.itp_id == 246:
        return "Caltrain"
    elif x.itp_id == 279:
        return "BART"
    else:
        return "Merced the Bus"

In [6]:
stops["operator"] = stops.apply(lambda x: operator_names(x), axis=1) 

#### Dictionary

In [7]:
agency_names = {279: 'BART', 246:'Caltrain', 343: 'Merced the Bus'}

In [8]:
stops["agency"] = stops.itp_id.map(agency_names)

In [9]:
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency
265688,246,2537740,37.438516,-122.156443,Stanford Caltrain,Caltrain,Caltrain
265689,246,2537740,37.438516,-122.156443,Stanford Caltrain Station,Caltrain,Caltrain


#### Checking that they are the same HELP
When I use assert it says "The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all()."

In [10]:
assert stops.operator == stops.agency

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [11]:
stops['operator'].equals(stops['agency']) #checking another way 

True

## Turn lat/lon into point geometry

There is a [function in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py#L170-L192) that does it. Show the steps within the function (the long way), and also create the `geometry` column using `shared_utils`.

In [12]:
#showing steps
stops2 = stops.assign(
      geometry = gpd.points_from_xy(stops['stop_lon'], stops['stop_lat'], 
      crs = 'WGS84'))

gdf = gpd.GeoDataFrame(stops2)

In [13]:
gdf.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency,geometry
265688,246,2537740,37.438516,-122.156443,Stanford Caltrain,Caltrain,Caltrain,POINT (-122.15644 37.43852)
265689,246,2537740,37.438516,-122.156443,Stanford Caltrain Station,Caltrain,Caltrain,POINT (-122.15644 37.43852)


In [14]:
gdf.shape

(1363, 8)

In [15]:
#using the function (for some reason it's not working)...

## Spatial Join (which points fall into which polygon)

This URL gives you CA county boundaries: https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12

* Go to "I want to use this" > View API Resources > copy link for geojson
* Read in the geojson with `geopandas` and make it a geodataframe: `gpd.read_file(LONG_URL_PATH)`
* Double check that the coordinate reference system is the same for both gdfs using `gdf.crs`. If not, change it so they are the same.
* Spatial join stops to counties: [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html)
    * Play with inner join or left join, what's the difference? Which one do you want?
    * Play with switching around the left_df and right_df, what's the right order?
* By county: count number of stops and stops per sq_mi.
    * Hint 1: Start with a CRS with units in feet or meters, then do a conversion to sq mi. [CRS in shared_utils](https://github.com/cal-itp/data-analyses/blob/main/_shared_utils/shared_utils/geography_utils.py)
    * Hint 2: to find area, you can create a new column and calculate `gdf.geometry.area`. [geometry manipulations docs](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html)

In [16]:
#Geo Json link: https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson

In [17]:
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson')

In [18]:
geojson.head(2)

,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry
0,1,Alameda,ALA,1,01,001,None,{E6F92268-D2DD-4CFB-8B79-5B4B2F07C559},2.538264,0.217411,"MULTIPOLYGON (((-122.27125 37.90503, -122.2702..."
1,2,Alpine,ALP,2,02,003,None,{870479B2-480A-494B-8352-AD60578839C1},2.170420,0.198471,"MULTIPOLYGON (((-119.58667 38.71420, -119.5865..."


#### Checking to see if the 2 dataframes have the same coordinate system, WGS 84 

In [19]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [20]:
geojson.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Spatial Join
I think the left join with my gpd data frame with the queried data & inner join are the same, just the columns are jumbled in different orders.

In [28]:
#inner
spatial_inner_data = gpd.sjoin(gdf, geojson, how='inner')

spatial_inner_data.shape

(1363, 19)

In [34]:
spatial_inner_data.head(3)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area
265688,246,2537740,37.438516,-122.156443,Stanford Caltrain,Caltrain,Caltrain,POINT (-122.15644 37.43852),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197
265689,246,2537740,37.438516,-122.156443,Stanford Caltrain Station,Caltrain,Caltrain,POINT (-122.15644 37.43852),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197
265690,246,2537744,37.438445,-122.156502,Stanford Caltrain Station,Caltrain,Caltrain,POINT (-122.15650 37.43845),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197


In [30]:
#left join 1
spatial_left1 = gpd.sjoin(gdf, geojson, how='left')

spatial_left1.shape

(1363, 19)

In [33]:
spatial_left1.head(3)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area
265688,246,2537740,37.438516,-122.156443,Stanford Caltrain,Caltrain,Caltrain,POINT (-122.15644 37.43852),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197
265689,246,2537740,37.438516,-122.156443,Stanford Caltrain Station,Caltrain,Caltrain,POINT (-122.15644 37.43852),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197
265690,246,2537744,37.438445,-122.156502,Stanford Caltrain Station,Caltrain,Caltrain,POINT (-122.15650 37.43845),42,43,Santa Clara,SCL,43,43,085,None,{2220F7B9-9361-4FB8-9456-872109CF6CF7},3.544078,0.343197


In [35]:
#checking to see if inner and first left join data frame are the same
spatial_left1.equals(spatial_inner_data)

False

In [38]:
#exporting to see what's going on
spatial_left1.to_csv("./spatial_left.csv", index= False)
spatial_inner_data.to_csv("./spatial_inner_data.csv", index= False)

In [31]:
#left join 2 switching data frames
spatial_left2 = gpd.sjoin(geojson,gdf, how='left')
spatial_left2.head(3)


,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry,index_right,itp_id,stop_id,stop_lat,stop_lon,stop_name,operator,agency
0,1,Alameda,ALA,1,01,001,None,{E6F92268-D2DD-4CFB-8B79-5B4B2F07C559},2.538264,0.217411,"MULTIPOLYGON (((-122.27125 37.90503, -122.2702...",310539.0,279.0,place_WARM,37.502285,-121.939395,Warm Springs / South Fremont,BART,BART
0,1,Alameda,ALA,1,01,001,None,{E6F92268-D2DD-4CFB-8B79-5B4B2F07C559},2.538264,0.217411,"MULTIPOLYGON (((-122.27125 37.90503, -122.2702...",310387.0,279.0,WARM,37.502285,-121.939395,Warm Springs / South Fremont,BART,BART
0,1,Alameda,ALA,1,01,001,None,{E6F92268-D2DD-4CFB-8B79-5B4B2F07C559},2.538264,0.217411,"MULTIPOLYGON (((-122.27125 37.90503, -122.2702...",310390.0,279.0,WARM,37.502285,-121.939395,Warm Springs / South Fremont,BART,BART


In [32]:
#this has way more columns..
spatial_left2.shape

(1425, 19)

In [39]:
spatial_left2.to_csv("./spatial_left2.csv", index= False) #the export looks totally crazy and WAY too many rows...

#### By county: count number of stops and stops per sq_mi

In [41]:
spatial_inner_data = spatial_inner_data.to_crs({'init':'epsg:2229'})

/opt/conda/lib/python3.9/site-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [42]:
spatial_inner_data.crs


<Derived Projected CRS: EPSG:2229>
Name: NAD83 / California zone 5 (ftUS)
Axis Info [cartesian]:
- E[east]: Easting (US survey foot)
- N[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - California - counties Kern; Los Angeles; San Bernardino; San Luis Obispo; Santa Barbara; Ventura.
- bounds: (-121.42, 32.76, -114.12, 35.81)
Coordinate Operation:
- name: SPCS83 California zone 5 (US Survey feet)
- method: Lambert Conic Conformal (2SP)
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich